This script classifies each new housing development area according to the dominant building type by comparing the footprint areas of residential and non-residential new buildings. Only residential-dominant development areas are retained, assigned unique district-based IDs, and exported for further analysis.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# =============================================================================
# CONFIGURATION
# =============================================================================

NHDA_PATH   = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\New_Housing_Development_Areas\New_Housing_Development_Areas_new_buildup_ratio1_dissolved.gpkg"
LOD2_NEW    = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\LoD2_2025_new_buildings.gpkg"
VG250_PATH  = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\Verwaltungsgebiete\vg250-ew_12-31.utm32s.gpkg.ebenen\vg250-ew_ebenen_1231\DE_VG250.gpkg"
VG250_LAYER = "vg250_krs"
OUTPUT_DIR  = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\New_Housing_Development_Areas"

CLASS_COL       = "building_class"
RESIDENTIAL_VAL = "residential"

TARGET_CRS = "EPSG:25832"

# =============================================================================
# MAIN
# =============================================================================

def main():
    print("\n" + "="*70)
    print("NEW HOUSING DEVELOPMENT AREAS — DOMINANT CLASS CLASSIFICATION")
    print("="*70)

    output_path = Path(OUTPUT_DIR)
    output_path.mkdir(parents=True, exist_ok=True)

    for label, path in [("NHDA", NHDA_PATH), ("LoD2 New", LOD2_NEW), ("VG250", VG250_PATH)]:
        exists = Path(path).exists()
        print(f"{'Check' if exists else 'x'} {label}: {Path(path).name}")
        if not exists:
            return

    start = datetime.now()
    print(f"\nStart: {start.strftime('%Y-%m-%d %H:%M:%S')}\n")

    # =========================================================================
    # STEP 1: Load data
    # =========================================================================
    print("="*60)
    print("STEP 1: Load data")
    print("="*60)

    gdf_nhda   = gpd.read_file(NHDA_PATH).to_crs(TARGET_CRS)
    gdf_new    = gpd.read_file(LOD2_NEW).to_crs(TARGET_CRS)
    landkreise = gpd.read_file(VG250_PATH, layer=VG250_LAYER).to_crs(TARGET_CRS)
    landkreise = landkreise[landkreise["AGS"].astype(str).str.startswith("09")].copy()

    print(f"NHDA polygons:       {len(gdf_nhda):,}")
    print(f"New buildings:       {len(gdf_new):,}")
    print(f"Districts (Bavaria): {len(landkreise):,}")
    print(f"Building classes:    {gdf_new[CLASS_COL].value_counts().to_dict()}")

    gdf_nhda = gdf_nhda.reset_index(drop=True)
    gdf_nhda['nhda_idx'] = gdf_nhda.index

    # =========================================================================
    # STEP 2: Split new buildings into residential vs non-residential
    # =========================================================================
    print(f"\n{'='*60}")
    print("STEP 2: Split new buildings by class")
    print("="*60)

    gdf_res    = gdf_new[gdf_new[CLASS_COL] == RESIDENTIAL_VAL].copy()
    gdf_nonres = gdf_new[gdf_new[CLASS_COL] != RESIDENTIAL_VAL].copy()

    print(f"Residential:         {len(gdf_res):,}")
    print(f"Non-res + auxiliary: {len(gdf_nonres):,}")

    # =========================================================================
    # STEP 3: Calculate footprint area per NHDA polygon
    # =========================================================================
    print(f"\n{'='*60}")
    print("STEP 3: Calculate footprint areas per polygon")
    print("="*60)

    def footprint_per_polygon(gdf_buildings, gdf_areas, col_name):
        if len(gdf_buildings) == 0:
            return pd.Series(0.0, index=gdf_areas['nhda_idx'], name=col_name)

        areas     = gdf_areas[['geometry', 'nhda_idx']].copy()
        buildings = gdf_buildings[['geometry']].copy()

        overlay = gpd.overlay(buildings, areas, how='intersection', keep_geom_type=False)
        overlay['area_m2'] = overlay.geometry.area

        result = (
            overlay.groupby('nhda_idx')['area_m2']
            .sum()
            .rename(col_name)
        )
        return result

    print("   Computing residential footprint...")
    res_fp    = footprint_per_polygon(gdf_res,    gdf_nhda, 'res_footprint_m2')

    print("   Computing non-residential footprint...")
    nonres_fp = footprint_per_polygon(gdf_nonres, gdf_nhda, 'nonres_footprint_m2')

    # =========================================================================
    # STEP 4: Merge back and compute ratios
    # =========================================================================
    print(f"\n{'='*60}")
    print("STEP 4: Compute ratios & assign dominant class")
    print("="*60)

    gdf_out = gdf_nhda.copy()
    gdf_out = gdf_out.merge(res_fp.reset_index(),    on='nhda_idx', how='left')
    gdf_out = gdf_out.merge(nonres_fp.reset_index(), on='nhda_idx', how='left')

    gdf_out['res_footprint_m2']    = gdf_out['res_footprint_m2'].fillna(0).round(2)
    gdf_out['nonres_footprint_m2'] = gdf_out['nonres_footprint_m2'].fillna(0).round(2)

    total = gdf_out['res_footprint_m2'] + gdf_out['nonres_footprint_m2']

    gdf_out['res_ratio']    = np.where(total > 0, gdf_out['res_footprint_m2']    / total, np.nan).round(4)
    gdf_out['nonres_ratio'] = np.where(total > 0, gdf_out['nonres_footprint_m2'] / total, np.nan).round(4)

    def assign_dominant(row):
        if pd.isna(row['res_ratio']):
            return 'no_new_buildings'
        if row['res_footprint_m2'] > row['nonres_footprint_m2']:
            return 'residential'
        elif row['nonres_footprint_m2'] > row['res_footprint_m2']:
            return 'non_residential'
        else:
            return 'equal'

    gdf_out['dominant_class'] = gdf_out.apply(assign_dominant, axis=1)
    gdf_out = gdf_out.drop(columns=['nhda_idx'], errors='ignore')

    # =========================================================================
    # STEP 5: Filter to residential-dominant only
    # =========================================================================
    print(f"\n{'='*60}")
    print("STEP 5: Filter to residential-dominant areas only")
    print("="*60)

    print(f"\nDominant class distribution (before filter):")
    for cls, cnt in gdf_out['dominant_class'].value_counts().items():
        pct = cnt / len(gdf_out) * 100
        print(f"  {cls:<20} {cnt:>5}  ({pct:.1f}%)")

    before  = len(gdf_out)
    gdf_out = gdf_out[gdf_out['dominant_class'] == 'residential'].copy().reset_index(drop=True)
    print(f"\n→ Kept: {len(gdf_out):,} / {before:,} polygons "
          f"({len(gdf_out)/before*100:.1f}% residential-dominant)")

    # =========================================================================
    # STEP 6: Spatial Join → District AGS + numbering
    # =========================================================================
    print(f"\n{'='*60}")
    print("STEP 6: Assign district AGS & generate nhda_id")
    print("="*60)

    # Centroid-based join (more stable than polygon-on-polygon)
    centroids = gdf_out.copy()
    centroids["geometry"] = gdf_out.geometry.centroid

    joined = gpd.sjoin(
        centroids[["geometry"]],
        landkreise[["AGS", "GEN", "geometry"]],
        how="left",
        predicate="within"
    ).rename(columns={"AGS": "lk_ags", "GEN": "lk_name"})

    # Edge polygons → nearest
    missing = joined["lk_ags"].isna()
    if missing.sum() > 0:
        print(f"   {missing.sum()} centroids outside → nearest district...")
        nearest = gpd.sjoin_nearest(
            centroids.loc[joined[missing].index, ["geometry"]],
            landkreise[["AGS", "GEN", "geometry"]],
            how="left"
        ).rename(columns={"AGS": "lk_ags", "GEN": "lk_name"})
        nearest = nearest[~nearest.index.duplicated(keep="first")]
        joined.loc[missing, "lk_ags"]  = nearest["lk_ags"]
        joined.loc[missing, "lk_name"] = nearest["lk_name"]

    gdf_out["lk_ags"]  = joined["lk_ags"].astype(str).values
    gdf_out["lk_name"] = joined["lk_name"].values

    # Number sequentially per district → nhda_id: AGS_1, AGS_2, ...
    gdf_out = gdf_out.sort_values("lk_ags").reset_index(drop=True)
    gdf_out["nhda_id"] = (
        gdf_out.groupby("lk_ags").cumcount().add(1).astype(str)
    )
    gdf_out["nhda_id"] = gdf_out["lk_ags"] + "_" + gdf_out["nhda_id"]

    print(f"   Districts with NHDA polygons: {gdf_out['lk_ags'].nunique()}")
    print(f"   Example IDs: {gdf_out['nhda_id'].head(5).tolist()}")

    # =========================================================================
    # STEP 7: Summary & export
    # =========================================================================
    print(f"\n{'='*60}")
    print("STEP 7: Summary & export")
    print("="*60)

    print(f"\nRes ratio statistics:")
    r = gdf_out['res_ratio']
    print(f"  Median={r.median():.2f}  P25={r.quantile(0.25):.2f}  "
          f"P75={r.quantile(0.75):.2f}  Min={r.min():.2f}  Max={r.max():.2f}")

    print(f"\nNHDA per district (Top 10):")
    for ags, cnt in gdf_out['lk_ags'].value_counts().head(10).items():
        name = gdf_out.loc[gdf_out['lk_ags'] == ags, 'lk_name'].iloc[0]
        print(f"  {ags}  {name:<30} {cnt:>4} polygons")

    out_path = Path(OUTPUT_DIR) / "New_Housing_Development_Areas_residential.gpkg"
    gdf_out.to_file(out_path, driver='GPKG', layer='New_Housing_Development_Areas_residential')
    print(f"\n✓ Saved: {out_path.name}")
    print(f"  New columns: res_footprint_m2, nonres_footprint_m2, "
          f"res_ratio, nonres_ratio, dominant_class, lk_ags, lk_name, nhda_id")
    print(f"\nDuration: {datetime.now() - start}")
    print("✓ DONE")


if __name__ == "__main__":
    main()